## PROJETO GREEN BELT: REDUÇÃO DA OCIOSIDADE DO CATÁLOGO DE PRODUTOS

### FASE 1: DEFINE (Definir)

#### 1.1 Declaração do Problema (Y - Crise)
O Catálogo Mestre de Medicamentos apresenta uma alta taxa de ociosidade, com **70% dos itens (Y)** nunca sendo utilizados em processos de compra nos últimos 12 meses. Isso gera ineficiência operacional, aumenta o custo de administração (COPQ) e prejudica a eficácia das buscas por itens válidos.

#### 1.2 Métrica Crítica (Y)
- **Nome:** Taxa de Não-Utilização (Ociosidade).
- **Unidade:** Porcentagem (%) de itens no `dim.produtos` que nunca aparecem em ordens de compra.
- **Baseline (Baseline):** 52,42%.

#### 1.3 Causa Raiz Hipotética (X)
- **Hipótese:** A Métrica Y é causada pela **Métrica X: Falha na Governabilidade do Item**, especificamente a ausência, incompletude ou invalidez do **Código ANVISA** no momento do cadastro do produto.

#### 1.4 Meta do Projeto (Melhoria)
- **Meta:** Reduzir a Taxa de Não-Utilização (Ociosidade) de **70% para 30%** em 6 meses, através da padronização e validação dos dados de governança (ANVISA).

In [5]:
# Célula 2: FASE MEASURE (Medir) - Carregamento e ETL

import pandas as pd
import numpy as np

# 1. Carregamento do Dimensão de Produtos
filepath = 'C:\\Users\\debor\\OneDrive\\Github\\GreenBelt\\data\\raw\\dim_produto.csv'
try:
    # Tentativa padrão
    df_catalogo = pd.read_csv(filepath)
    print("Sucesso: Dados carregados (método padrão).")
except FileNotFoundError:
    print("ERRO: Arquivo não encontrado. Verifique o nome do arquivo no diretório.")
    raise
except pd.errors.ParserError as e:
    # Caso haja linhas com número variável de campos, tentar leitura mais tolerante
    print("Aviso: ParserError ao ler CSV. Tentando leitura alternativa (engine='python', on_bad_lines='skip').")
    df_catalogo = pd.read_csv(filepath, engine='python', sep=None, on_bad_lines='skip')
    print("Sucesso: Dados carregados com leitura tolerante (linhas incorretas serão ignoradas).")

# 2. ETL Básico (Garantindo que a coluna ANVISA seja tratável)
# Se sua coluna ANVISA tiver outro nome, ajuste aqui:
# As variáveis COLUNA_ANVISA e COLUNA_STATUS_COMPRA já estão definidas no notebook; não as sobrescrevemos aqui.

# Verificar se a coluna ANVISA existe
if COLUNA_ANVISA not in df_catalogo.columns:
    raise KeyError(f"Coluna '{COLUNA_ANVISA}' não encontrada no arquivo. Colunas disponíveis: {list(df_catalogo.columns)}")

# Limpeza e preenchimento de nulos para a Métrica X (Causa Raiz)
df_catalogo[COLUNA_ANVISA] = df_catalogo[COLUNA_ANVISA].astype(str).str.strip().str.upper()
df_catalogo[COLUNA_ANVISA].replace(['NAN', 'NONE', ''], np.nan, inplace=True)

print(f"Total de Itens no Catálogo: {len(df_catalogo):,}")
print("Colunas disponíveis no seu dim.produtos:")
print(df_catalogo.dtypes)

df_catalogo.head()

Aviso: ParserError ao ler CSV. Tentando leitura alternativa (engine='python', on_bad_lines='skip').
Sucesso: Dados carregados com leitura tolerante (linhas incorretas serão ignoradas).


KeyError: "Coluna 'anvisa' não encontrada no arquivo. Colunas disponíveis: ['\\ufeffid_produto', 'codigo_br', 'descricao_catmat', 'generico', 'unidade_fornecimento']"

### Fase MEASURE (Medir) - Carregamento de Dados

In [ ]:
# Célula 2: FASE MEASURE (Medir) - Carregamento e ETL

import pandas as pd
import numpy as np

# 1. Carregamento do Dimensão de Produtos
try:
    # AJUSTE O NOME DO SEU ARQUIVO AQUI!
    df_catalogo = pd.read_csv('SEU_ARQUIVO_DIM_PRODUTOS.csv')
    print("Sucesso: Dados carregados.")
except FileNotFoundError:
    print("ERRO: Arquivo não encontrado. Verifique o nome do arquivo no diretório.")
    
# 2. ETL Básico (Garantindo que a coluna ANVISA seja tratável)
# Se sua coluna ANVISA tiver outro nome, ajuste aqui:
COLUNA_ANVISA = 'codigo_anvisa' # Ou 'ANVISA'
COLUNA_STATUS_COMPRA = 'status_utilizacao' # Ou 'STATUS'

# Limpeza e preenchimento de nulos para a Métrica X (Causa Raiz)
df_catalogo[COLUNA_ANVISA] = df_catalogo[COLUNA_ANVISA].astype(str).str.strip().str.upper()
df_catalogo[COLUNA_ANVISA].replace(['NAN', 'NONE', ''], np.nan, inplace=True)

print(f"Total de Itens no Catálogo: {len(df_catalogo):,}")
print("Colunas disponíveis no seu dim.produtos:")
print(df_catalogo.dtypes)

df_catalogo.head()

### Carregando a Tabela Fato (Compras)

In [ ]:
# Célula 3: FASE MEASURE (Medir) - Carregamento da Tabela Fato (Transações)

# O df_catalogo (dim.produtos) está carregado da Célula 2.

# 1. Carregamento da Tabela Fato de Compras
try:
    df_fatos = pd.read_csv('data/raw/fato_compras_medicamentos.csv')
    print("✅ Sucesso: Tabela Fato de Compras carregada.")
except FileNotFoundError:
    print("❌ ERRO: Arquivo da Tabela Fato não encontrado. Verifique o nome do arquivo.")

# 2. Visão Rápida da Fato para identificar a chave e o período
print(f"\nTotal de Transações de Compras: {len(df_fatos):,}")
print("-" * 30)
print("Colunas disponíveis na sua Fato (df_fatos.head()):")

df_fatos.head()

### FASE MEASURE (Medir) - Prova da Crise (Métrica Y)

In [ ]:
# Célula 4: FASE MEASURE (Medir) - Cálculo da Ociosidade (Métrica Y)

# 1. Identificar todos os produtos que foram utilizados
# Usamos o set() para obter IDs únicos da tabela fato (transações)
COLUNA_CHAVE_PRODUTO = 'id_produto'  # Ajuste se o nome da sua chave for diferente!

produtos_utilizados_ids = set(df_fatos[COLUNA_CHAVE_PRODUTO].unique())

# 2. Calcular a Métrica Y (Ociosidade/Utilização)
# Criar a coluna 'status_utilizacao' no df_catalogo (dim_produto)
df_catalogo['status_utilizacao'] = np.where(
    df_catalogo[COLUNA_CHAVE_PRODUTO].isin(produtos_utilizados_ids),
    'Utilizado',
    'Não Utilizado' # Métrica Y (Ociosidade)
)

# 3. Prova da Crise (Comprovação Estatística do Y)
analise_ociosidade = df_catalogo.groupby('status_utilizacao').agg(
    Contagem_Produtos=(COLUNA_CHAVE_PRODUTO, 'count')
).reset_index()

# Calcula as porcentagens
total_produtos = analise_ociosidade['Contagem_Produtos'].sum()
analise_ociosidade['% do Total'] = (analise_ociosidade['Contagem_Produtos'] / total_produtos) * 100

print("\n--- Prova da Crise (Métrica Y: Ociosidade do Catálogo) ---")
print(f"Total de Produtos Cadastrados (Baseline): {total_produtos:,}")

# Formatação profissional
analise_ociosidade['Contagem_Produtos'] = analise_ociosidade['Contagem_Produtos'].map('{:,}'.format)
analise_ociosidade['% do Total'] = analise_ociosidade['% do Total'].round(2).astype(str) + '%'

print("\n**Distribuição de Utilização do Catálogo (Fase MEASURE):**")
print(analise_ociosidade.to_string(index=False))

# Gerar o Insight final para o Project Charter
ociosidade_percentual = analise_ociosidade.loc[analise_ociosidade['status_utilizacao'] == 'Não Utilizado', '% do Total'].iloc[0]

print(f"\n✅ **CRÍTICO:** A Métrica Y está comprovada: {ociosidade_percentual} do catálogo está ocioso.")
print("O projeto segue para a Fase ANALYZE.")